In [1]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
torch.set_default_dtype(torch.float64)

R_EARTH_KM       = 6378.137
MU_EARTH_KM3_S2  = 398600.4418
J2_CONST         = 1.08262668e-3

DU = R_EARTH_KM
TU = np.sqrt((DU**3) / MU_EARTH_KM3_S2)
VU = DU / TU

MU_CANONICAL = 1.0
R_CANONICAL  = 1.0

def orbital_elements(r0, v0):
    r0n = torch.norm(r0, dim=-1, keepdim=True)
    if not ((r0n.min() > 0.9) & (r0n.max() < 20.0)):
        raise ValueError("r0 norm out of expected bounds.")
    v0n = torch.norm(v0, dim=-1, keepdim=True)
    v0_sq = v0n**2
    energy = 0.5 * v0_sq - MU_CANONICAL / r0n
    if (energy >= -1e-9).any():
        raise ValueError("Unbound IC not supported.")
    a = -MU_CANONICAL / (2.0 * energy)
    n = torch.sqrt(MU_CANONICAL / a**3)
    h2 = (r0**2).sum(-1, keepdim=True) * v0_sq - (r0 * v0).sum(-1, keepdim=True)**2
    ecc = torch.sqrt(torch.clamp(1.0 + 2.0 * energy * h2, min=0.0))
    return a, n, ecc, r0n, v0n

def canonical_accel(r):
    r_sq = (r**2).sum(dim=-1, keepdim=True).clamp_min(0.25)
    r_norm = torch.sqrt(r_sq)
    r_norm5 = r_norm**5
    z2 = (r[..., 2:3]**2) / r_sq
    fac_xy = 1.0 - 5.0 * z2
    fac_z  = 3.0 - 5.0 * z2
    fac = torch.cat([fac_xy, fac_xy, fac_z], dim=-1)
    a_kepler = -(MU_CANONICAL / (r_norm**3)) * r
    a_j2     = -1.5 * J2_CONST * (R_CANONICAL**2) * (r / r_norm5) * fac
    return a_kepler + a_j2

def rk4_reference(r0, v0, t_grid, dt=1e-4):
    r, v = r0.clone(), v0.clone()
    traj = [r.clone()]
    for i in range(len(t_grid) - 1):
        delta_t = (t_grid[i+1] - t_grid[i]).item()
        steps = max(1, int(delta_t / dt))
        h = delta_t / steps
        for _ in range(steps):
            k1r = v
            k1v = canonical_accel(r)
            k2r = v + 0.5 * h * k1v
            k2v = canonical_accel(r + 0.5 * h * k1r)
            k3r = v + 0.5 * h * k2v
            k3v = canonical_accel(r + 0.5 * h * k2r)
            k4r = v + h * k3v
            k4v = canonical_accel(r + h * k3r)
            r = r + (h / 6.0) * (k1r + 2*k2r + 2*k3r + k4r)
            v = v + (h / 6.0) * (k1v + 2*k2v + 2*k3v + k4v)
        traj.append(r.clone())
    return torch.stack(traj, dim=1)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

i_inc = np.radians(51.6)
r_leo = 7000.0 / DU
v_leo = np.sqrt(MU_CANONICAL / r_leo)

BATCH_SIZE = 512
r0_batch = torch.tensor([[r_leo, 0.0, 0.0]], device=device).repeat(BATCH_SIZE, 1)
v0_batch = torch.tensor(
    [[0.0, v_leo * np.cos(i_inc), v_leo * np.sin(i_inc)]], device=device
).repeat(BATCH_SIZE, 1)

_, n_leo_tensor, _, _, _ = orbital_elements(r0_batch[:1], v0_batch[:1])
n_leo = n_leo_tensor.item()
T_ORBIT = 2.0 * np.pi / n_leo
T_MAX = T_ORBIT * 1.02
print(f"n_kepler = {n_leo:.8f} /TU  |  T_orb = {T_ORBIT:.4f} TU  |  T_MAX = {T_MAX:.4f} TU")

# ==========================================================
# ===== FIXED (exact) ===== EMPIRICAL n_eff VIA EXACT SEGMENT ANGLES
# ==========================================================
with torch.no_grad():
    T_KEP = 2.0 * np.pi / n_leo
    N_SAMPLES = 2000
    t_check = torch.linspace(0, T_KEP, N_SAMPLES, device=device)
    r_check = rk4_reference(r0_batch[:1], v0_batch[:1], t_check, dt=1e-4).squeeze(0)

    r_seg_start = r_check[:-1]
    r_seg_end   = r_check[1:]

    # Exact angle between consecutive position vectors
    cross_vecs  = torch.cross(r_seg_start, r_seg_end, dim=-1)
    cross_mag   = torch.norm(cross_vecs, dim=-1)
    dot_vals    = (r_seg_start * r_seg_end).sum(dim=-1)
    dtheta      = torch.atan2(cross_mag, dot_vals)     # (N-1,) all in [0, pi]

    theta_total = dtheta.sum().item()
    n_eff_emp   = theta_total / T_KEP
    DELTA_N_OVER_N = (n_eff_emp - n_leo) / n_leo

analytic_dn = 0.75 * J2_CONST * (1.0 / r_leo)**2 * (3.0 * np.cos(i_inc)**2 - 1.0)
print(f"theta_total (one Kepler period) = {theta_total:.8f} rad")
print(f"  expected 2*pi = {2*np.pi:.8f}")
print(f"  excess = {theta_total - 2*np.pi:+.6e} rad")
print(f"n_eff_empirical = {n_eff_emp:.10f} /TU")
print(f"DELTA_N_OVER_N empirical = {DELTA_N_OVER_N:+.6e}")
print(f"DELTA_N_OVER_N analytic  = {analytic_dn:+.6e}")
print(f"Ratio empirical/analytic = {DELTA_N_OVER_N / analytic_dn:.4f}")

# ==========================================================
class OrbitalDeepONet(nn.Module):
    def __init__(self, branch_dim=6, trunk_dim=1, hidden_dim=128):
        super().__init__()
        self.branch = nn.Sequential(
            nn.Linear(branch_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3 * hidden_dim)
        )
        self.trunk = nn.Sequential(
            nn.Linear(trunk_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3 * hidden_dim)
        )
        self.hidden_dim = hidden_dim

    def forward(self, r0, v0, t):
        _, n, _, _, _ = orbital_elements(r0, v0)
        n_eff = n * (1.0 + DELTA_N_OVER_N)
        ic = torch.cat([r0, v0], dim=-1)
        b_out = self.branch(ic).view(-1, 3, self.hidden_dim)
        t_out = self.trunk(t).view(-1, 3, self.hidden_dim)
        nn_out = torch.sum(b_out * t_out, dim=-1) / np.sqrt(self.hidden_dim)
        c = torch.cos(n_eff * t)
        s = torch.sin(n_eff * t)
        m = (1.0 - torch.exp(-t))**2
        return r0 * c + (v0 / n_eff) * s + m * nn_out

def wrap_to_pi(x):
    return (x + torch.pi) % (2.0 * torch.pi) - torch.pi

def compute_physics_loss(model, r0, v0, t, raan_weight=1e2, escape_weight=1e-3):
    t_eval = t.clone().detach().requires_grad_(True)
    r_pred = model(r0, v0, t_eval)

    d1_list, d2_list = [], []
    for i in range(3):
        comp = r_pred[:, i:i+1]
        grad_1 = torch.autograd.grad(
            comp.sum(), t_eval, create_graph=True, retain_graph=True
        )[0]
        grad_2 = torch.autograd.grad(
            grad_1.sum(), t_eval, create_graph=True
        )[0]
        d1_list.append(grad_1)
        d2_list.append(grad_2)

    v_pred = torch.cat(d1_list, dim=-1)
    a_pred = torch.cat(d2_list, dim=-1)

    a_true   = canonical_accel(r_pred)
    residual = a_pred - a_true
    r_mag_sq = (r_pred.detach()**2).sum(dim=-1, keepdim=True)
    physics_loss = torch.mean((residual * r_mag_sq)**2)

    h_vec      = torch.cross(r_pred, v_pred, dim=-1)
    Omega_pred = torch.atan2(h_vec[..., 0:1], -h_vec[..., 1:2])
    h0_vec = torch.cross(r0, v0, dim=-1)
    Omega_0 = torch.atan2(h0_vec[..., 0:1], -h0_vec[..., 1:2])

    a_el, n_el, ecc_el, r0n_el, v0n_el = orbital_elements(r0, v0)
    h0_norm = torch.norm(h0_vec, dim=-1, keepdim=True)
    inc = torch.acos(torch.clamp(h0_vec[..., 2:3] / h0_norm, -1.0, 1.0))
    Omega_dot_true = -1.5 * J2_CONST * n_el * (R_CANONICAL / a_el)**2 * torch.cos(inc)

    Omega_target   = Omega_0 + Omega_dot_true * t_eval
    Omega_residual = wrap_to_pi(Omega_pred - Omega_target)
    raan_loss      = torch.mean(Omega_residual**2)

    M = r0n_el**2 + (v0n_el / n_el)**2
    D = torch.sqrt(
        (r0n_el**2 - (v0n_el / n_el)**2)**2
        + 4.0 * (r0 * v0).sum(-1, keepdim=True)**2 / n_el**2
    )
    base_amp = torch.sqrt(0.5 * (M + D))
    r_barrier = torch.maximum(a_el * (1.0 + ecc_el), base_amp) + 0.5
    r_norm    = torch.norm(r_pred, dim=-1, keepdim=True)
    escape_penalty = torch.mean(
        torch.clamp(torch.relu(r_norm - r_barrier), max=5.0)**4
    )

    total_loss = physics_loss + (raan_weight * raan_loss) + (escape_weight * escape_penalty)
    return total_loss, physics_loss, raan_loss, escape_penalty, r_norm

model     = OrbitalDeepONet(hidden_dim=256).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS    = 75000
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-5
)

t_grid    = torch.linspace(0, T_MAX, 200, device=device).unsqueeze(-1)
t_grid_1d = t_grid.squeeze(-1)

with torch.no_grad():
    r_ref   = rk4_reference(r0_batch[:1], v0_batch[:1], t_grid_1d, dt=1e-4).squeeze(0)
    r0_eval = r0_batch[:1].repeat(200, 1)
    v0_eval = v0_batch[:1].repeat(200, 1)

print(f"Initiating Empirical-n_eff Run (Batch={BATCH_SIZE}, {EPOCHS} Epochs)...")

for epoch in range(1, EPOCHS + 1):

    if epoch < 20000:
        current_raan_wt = 100.0
    elif epoch < 30000:
        current_raan_wt = 100.0 * (30000 - epoch) / 10000.0
    else:
        current_raan_wt = 1.0

    optimizer.zero_grad()
    t_batch = torch.rand(BATCH_SIZE, 1, device=device) * T_MAX

    total_loss, physics_loss, raan_loss, escape_penalty, r_norm = \
        compute_physics_loss(model, r0_batch, v0_batch, t_batch, raan_weight=current_raan_wt)

    is_log_epoch = (epoch % 1000 == 0 or epoch == 1)
    need_diag    = is_log_epoch and current_raan_wt > 0.0

    total_loss.backward(retain_graph=need_diag)

    gnorm_sq = torch.zeros((), device=device)
    for p in model.parameters():
        if p.grad is not None:
            gnorm_sq = gnorm_sq + p.grad.detach().norm(2) ** 2
    grad_norm_val = gnorm_sq.sqrt().item()

    g_phys_val, g_raan_val, cos_sim = 0.0, 0.0, 0.0
    if need_diag:
        saved = {n: p.grad.detach().clone() for n, p in model.named_parameters() if p.grad is not None}
        optimizer.zero_grad()
        physics_loss.backward(retain_graph=True)
        g_phys_val = torch.sqrt(sum(p.grad.detach().norm(2)**2 for p in model.parameters() if p.grad is not None)).item()
        phys_grad_vec = torch.cat([p.grad.detach().flatten() for p in model.parameters() if p.grad is not None])
        optimizer.zero_grad()
        (current_raan_wt * raan_loss).backward()
        g_raan_val = torch.sqrt(sum(p.grad.detach().norm(2)**2 for p in model.parameters() if p.grad is not None)).item()
        raan_grad_vec = torch.cat([p.grad.detach().flatten() for p in model.parameters() if p.grad is not None])
        if g_phys_val > 0 and g_raan_val > 0:
            cos_sim = torch.nn.functional.cosine_similarity(phys_grad_vec, raan_grad_vec, dim=0).item()
        for n, p in model.named_parameters():
            if n in saved:
                p.grad = saved[n]

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
    optimizer.step()
    scheduler.step()

    if is_log_epoch:
        with torch.no_grad():
            r_pred_grid = model(r0_eval, v0_eval, t_grid)
            pos_err_km  = torch.norm(r_pred_grid - r_ref, dim=-1).max().item() * DU

            r_hat      = r_ref / torch.norm(r_ref, dim=-1, keepdim=True).clamp_min(1e-12)
            err_vec    = r_pred_grid - r_ref
            err_radial = (err_vec * r_hat).sum(-1, keepdim=True) * r_hat
            err_rest   = err_vec - err_radial
            alongtrack_km = torch.norm(err_rest, dim=-1).max().item() * DU

            dt = (t_grid_1d[1] - t_grid_1d[0]).item()
            v_pred_grid = (r_pred_grid[2:] - r_pred_grid[:-2]) / (2.0 * dt)
            r_mid       = r_pred_grid[1:-1]
            h_grid      = torch.cross(r_mid, v_pred_grid, dim=-1)
            Omega_grid  = torch.atan2(h_grid[..., 0:1], -h_grid[..., 1:2])

            h0e  = torch.cross(r0_eval[:1], v0_eval[:1], dim=-1)
            Omega_0_eval = torch.atan2(h0e[..., 0:1], -h0e[..., 1:2])
            a_e, n_e, _, _, _ = orbital_elements(r0_eval[:1], v0_eval[:1])
            h0n = torch.norm(h0e, dim=-1, keepdim=True)
            inc_e = torch.acos(torch.clamp(h0e[..., 2:3] / h0n, -1.0, 1.0))
            Omega_dot_e = -1.5 * J2_CONST * n_e * (R_CANONICAL / a_e)**2 * torch.cos(inc_e)

            Omega_target_grid = Omega_0_eval + Omega_dot_e * t_grid[1:-1]
            raan_err_deg = wrap_to_pi(Omega_grid - Omega_target_grid).abs().max().item() * 180.0 / np.pi

            print(
                f"Ep {epoch:5d} | "
                f"Phys: {physics_loss.item():.2e} | "
                f"RAAN: {raan_loss.item():.2e} | "
                f"λR: {current_raan_wt:.1f} | "
                f"ΔΩ: {raan_err_deg:.4f}° | "
                f"err: {pos_err_km:.3f} km | "
                f"along: {alongtrack_km:.3f} km"
            )

n_kepler = 0.86974853 /TU  |  T_orb = 7.2241 TU  |  T_MAX = 7.3686 TU
theta_total (one Kepler period) = 6.29232137 rad
  expected 2*pi = 6.28318531
  excess = +9.136059e-03 rad
n_eff_empirical = 0.8710131906 /TU
DELTA_N_OVER_N empirical = +1.454049e-03
DELTA_N_OVER_N analytic  = +1.061548e-04
Ratio empirical/analytic = 13.6974
Initiating Empirical-n_eff Run (Batch=512, 75000 Epochs)...
Ep     1 | Phys: 2.71e-04 | RAAN: 2.78e-04 | λR: 100.0 | ΔΩ: 21.8998° | err: 2944.242 km | along: 2680.388 km
Ep  1000 | Phys: 1.21e-06 | RAAN: 4.27e-08 | λR: 100.0 | ΔΩ: 0.0275° | err: 16.799 km | along: 16.798 km
Ep  2000 | Phys: 5.10e-07 | RAAN: 1.29e-08 | λR: 100.0 | ΔΩ: 0.0152° | err: 14.037 km | along: 14.027 km
Ep  3000 | Phys: 5.14e-07 | RAAN: 4.14e-08 | λR: 100.0 | ΔΩ: 0.0238° | err: 14.378 km | along: 14.378 km
Ep  4000 | Phys: 5.07e-07 | RAAN: 1.47e-08 | λR: 100.0 | ΔΩ: 0.0128° | err: 13.459 km | along: 13.459 km
Ep  5000 | Phys: 5.71e-07 | RAAN: 1.16e-08 | λR: 100.0 | ΔΩ: 0.0157° | err: 13.56